# Inspect FAISS Index

Inspect `data/faiss_index.bin` and map vector ids to rows in `data/trading.db`.

In [ ]:
from pathlib import Path
import sqlite3

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent

FAISS_PATH = ROOT / 'data' / 'faiss_index.bin'
DB_PATH = ROOT / 'data' / 'trading.db'
print('ROOT      :', ROOT)
print('FAISS PATH:', FAISS_PATH, 'exists=', FAISS_PATH.exists())
print('DB PATH   :', DB_PATH, 'exists=', DB_PATH.exists())

In [ ]:
import faiss

index = faiss.read_index(str(FAISS_PATH))
print('index type:', type(index).__name__)
print('ntotal    :', index.ntotal)
print('dimension :', index.d)

In [ ]:
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row

tables = [r['name'] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY 1")]
print('tables:', tables)

def preview_table(name, limit=5):
    try:
        rows = conn.execute(f"SELECT * FROM {name} LIMIT ?", (limit,)).fetchall()
        print(f"\\n[{name}] rows={len(rows)}")
        for row in rows:
            print(dict(row))
    except Exception as e:
        print(f"[{name}] preview failed: {e}")

for candidate in ['signal_history', 'signals', 'trades']:
    if candidate in tables:
        preview_table(candidate)

In [ ]:
# Optional: inspect first few vectors
import numpy as np

n = min(3, index.ntotal)
if n > 0:
    ids = np.arange(n, dtype='int64')
    vecs = np.vstack([index.reconstruct(int(i)) for i in ids])
    print('reconstructed shape:', vecs.shape)
    print('first vector (first 10 values):', vecs[0][:10])
else:
    print('index is empty')